In [2]:
import os 
import warnings 

import pandas as pd
import numpy as np

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

/Users/op24226/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/op24226/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the datasets
data_path = '/Users/op24226/Desktop/EssayTB2/Data'

train = pd.read_csv(os.path.join(data_path, 'train.tsv'), sep='\t')
test = pd.read_csv(os.path.join(data_path, 'test.tsv'), sep='\t', index_col=0).reset_index(drop=True)
dev = pd.read_csv(os.path.join(data_path, 'dev.tsv'), sep='\t')

# Combine the datasets
combined = pd.concat([train, dev, test], ignore_index=True)

# Sample from the combined dataset and include true labels from the dataset
labels = {'true', 'false', 'mixture', 'unproven'}
pool = combined[combined['label'].isin(labels)].copy()

N = 500
sample = pool.sample(n = N, random_state = 42).reset_index(drop=True)

In [4]:
# Get LLM predictions
model = "google/flan-t5-base" 

print(f"Loading model...:{model}")

tokenizer = AutoTokenizer.from_pretrained(model)

model = AutoModelForSeq2SeqLM.from_pretrained(model)

generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

Loading model...:google/flan-t5-base


Device set to use mps:0


In [5]:
# Build prompt
llm_predictions = []

for i, claim in enumerate(sample["claim"]):
    prompt = (
        f"You are doing public health fact-checking.\n"
        f"Classify the following public health claim into one of these "
        f"categories: true, false, mixture, unproven.\n"
        f"- true: the claim is supported by evidence\n"
        f"- false: the claim is contradicted by evidence\n"
        f"- mixture: the claim is partially true and partially false\n"
        f"- unproven: there is insufficient evidence\n\n"
        f"Rules: "
        f"Use exactly one label from: true, false, mixture, unproven\n"
        f"Keep the reason short and plain\n"
        f"Do not add anything else\n\n"
        f"Claim: {claim}\n\nCategory:"
    )

    output = generator(prompt, max_new_tokens=15)[0]["generated_text"].strip().lower()

    pred = "unknown"
    for label in labels:
        if label in output:
            pred = label
            break

    llm_predictions.append(pred)

    if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(sample)} claims...")

sample["llm_prediction"] = llm_predictions
print(f"LLM predictions complete.")

Processed 50/500 claims...
Processed 100/500 claims...
Processed 150/500 claims...
Processed 200/500 claims...
Processed 250/500 claims...
Processed 300/500 claims...
Processed 350/500 claims...
Processed 400/500 claims...
Processed 450/500 claims...
Processed 500/500 claims...
LLM predictions complete.


In [6]:
# Create correctness labels
sample["ground_truth_text"] = sample["label"]

sample["llm_correct"] = (sample["llm_prediction"] == sample["ground_truth_text"]).astype(int)

print(f"Overall LLM accuracy: {sample['llm_correct'].mean():.1%}")

print(f"Correct: {sample['llm_correct'].sum()}, "
      f"Wrong: {(~sample['llm_correct'].astype(bool)).sum()}")

print(f"\nAccuracy by ground-truth category:")
for cat in labels:
    mask = sample["ground_truth_text"] == cat
    if mask.any():
        a = sample.loc[mask, "llm_correct"].mean()
        print(f"    {cat:12s}: {a:.1%}  (n={mask.sum()})")

Overall LLM accuracy: 40.6%
Correct: 203, Wrong: 297

Accuracy by ground-truth category:
    mixture     : 0.0%  (n=57)
    false       : 54.7%  (n=159)
    unproven    : 0.0%  (n=23)
    true        : 44.4%  (n=261)


In [7]:
print(pd.crosstab(sample["label"], sample["llm_prediction"], normalize="index").round(2))

llm_prediction  false  mixture  true  unproven
label                                         
false            0.55     0.02  0.38      0.06
mixture          0.37     0.00  0.60      0.04
true             0.47     0.00  0.44      0.09
unproven         0.74     0.04  0.22      0.00


In [8]:
sample.to_csv("llm_outputs.csv")

In [9]:
sample.head()

,claim_id,claim,date_published,explanation,fact_checkers,main_text,sources,label,subjects,llm_prediction,ground_truth_text,llm_correct
0,7962,Coronavirus can spread one to three days befor...,"April 1, 2020",People infected with the novel coronavirus can...,,"The study, which underscored the importance of...",,true,Health News,true,true,1
1,21353,We have had over 40 days this summer where it ...,"September 30, 2011",How dangerous is it to breathe Atlanta air?,Eric Stirgus,"""For many Atlanta-area residents, 2011 will be...",http://www.gaepd.org/air/smogforecast/stats.php,mixture,"Georgia, Environment, Transportation, Jennette...",false,mixture,0
2,15365,Obama keeps talking about getting rid of all n...,"July 17, 2015","""Cheney said, """"Obama keeps talking about gett...",Katie Sanders,"""Fox News host Sean Hannity found a partner in...",https://www.politifact.com/truth-o-meter/state...,false,"Nuclear, PunditFact, Dick Cheney,",false,false,1
3,7691,China to extend winter anti-smog measures for ...,"March 6, 2019",China will extend winter anti-smog measures su...,,The Ministry of Ecology and Environment (MEE) ...,,true,Environment,true,true,1
4,2830,More infected in fresh wave of China bird flu ...,"January 20, 2014",Another 23 people in China have been infected ...,Kate Kelland,With many people travelling within the country...,,true,Health News,false,true,0
